<a href="https://colab.research.google.com/github/jalammar/jalammar.github.io/blob/master/notebooks/distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
import torch

In [ ]:
!pip install transformers

In [ ]:
import transformers as ppb

In [ ]:
df = pd.read_csv('https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv', delimiter='\t', header=None)

In [ ]:
batch_1 = df[:2000]

In [ ]:
batch_1[1].value_counts()

In [ ]:
model_class, tokenizer_class, pretrained_weights = (ppb.DistilBertModel, ppb.DistilBertTokenizer, 'distilbert-base-uncased')


# Load pretrained model/tokenizer
tokenizer = tokenizer_class.from_pretrained(pretrained_weights)
model = model_class.from_pretrained(pretrained_weights)

In [ ]:
tokenized = batch_1[0].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))

In [ ]:
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)
max_len

In [ ]:
padded = [i + [0]*(max_len-len(i)) for i in tokenized.values]

In [ ]:
np.array(padded).shape

In [ ]:
input_ids = torch.tensor(np.array(padded))  # Add special tokens takes care of adding [CLS], [SEP], <s>... tokens in the right way for each model.

with torch.no_grad():
    last_hidden_states = model(input_ids)

In [ ]:
features = last_hidden_states[0][:,0,:].numpy()

In [ ]:
labels = batch_1[1]

In [ ]:
from sklearn.model_selection import train_test_split
train_features, test_features, train_labels, test_labels = train_test_split(features, labels)

In [ ]:
from sklearn.linear_model import LogisticRegression
lr_clf = LogisticRegression()
# lr_clf.fit(train_features, train_labels)

scores = cross_val_score(lr_clf, train_features, train_labels)
print("Logistic Regression classifier score: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

In [ ]:
lr_clf = LogisticRegression()
lr_clf.fit(train_features, train_labels)
lr_clf.score(test_features, test_labels)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
random_forest = RandomForestClassifier()
scores = cross_val_score(random_forest, train_features, train_labels)
print("Random forest classifier score: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))


In [ ]:
from sklearn.svm import SVC
svc_regressor = SVC()

scores = cross_val_score(svc_regressor, train_features, train_labels)
print("SVM classifier average score: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
# est = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,max_depth=1, random_state=0, loss='ls')
gbc = GradientBoostingClassifier()

scores = cross_val_score(gbc, train_features, train_labels, cv=5)
print("Gradient Boosting classifier accuracy: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))



In [ ]:
from sklearn.neural_network import MLPClassifier
neural_network = MLPClassifier(hidden_layer_sizes=(1000, 300), learning_rate='adaptive', verbose=False)
neural_network.fit(train_features, train_labels)
neural_network.score(test_features, test_labels)
# scores = cross_val_score(neural_network, train_features, train_labels)
# print("Neural network regressor score: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))


In [ ]:
from sklearn.neural_network import MLPClassifier
neural_network = MLPClassifier(hidden_layer_sizes=(2000), learning_rate='adaptive', verbose=False)
neural_network.fit(train_features, train_labels)
neural_network.score(test_features, test_labels)


In [ ]:
from sklearn.dummy import DummyClassifier
clf = DummyClassifier()

scores = cross_val_score(clf, train_features, train_labels)
print("Dummy classifier score: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('500_with_vectors.csv')
df = df.drop(['Unnamed: 0'], axis=1)

In [ ]:
df